In [1]:
import os
import sys
from pathlib import Path
PROJECT_ROOT = Path("..").resolve()
sys.path.append(str(PROJECT_ROOT))
from omegaconf import OmegaConf
from functools import partial

import numpy as np
import torch
from torchtune import config
from torchtune.data import padded_collate_packed
from torch.utils.data import DataLoader
import torch.nn.functional as F


from dataset_classes import learning_levels_pfa_dataset, PackedOnTheFlyDataset
from evaluation.pfa_evaluation import generate_per_token_losses, decode_token_by_token
from training import SelfPredictionTrainingRecipeDistributed

cfg = OmegaConf.load(f'{PROJECT_ROOT}/configs/llama_0.1B_PHi.yaml')

In [2]:
cfg_tokenizer = cfg.tokenizer
tokenizer = config.instantiate(cfg.tokenizer)

In [3]:
included_levels = [0,2]

cfg_dataset = cfg.dataset
cfg_dataset.included_learning_levels = included_levels
packed_on_the_fly = cfg_dataset.pop("packed_on_the_fly", False)
packed_sequence_length = cfg_dataset.pop("packed_sequence_length", 2048)
split_across_pack = cfg_dataset.pop("split_across_pack", False)
num_workers = cfg_dataset.pop("num_workers", 8)

ds = config.instantiate(cfg_dataset, tokenizer)
sample_ds = next(iter(ds))

In [4]:
sample_ds.keys()

dict_keys(['tokens', 'labels', 'new_language', 'learning_level', 'num_states', 'num_edges', 'vocab_size', 'perturbation', '_transition_probs', '_transition_symbols'])

In [5]:
base_dir = '/home/woody/iwbi/iwbi106h/suuraj/models/self-prediction-models'
model_path = 'learning_levels_sweep_x1bmwa36'

base_model_path = os.path.join(base_dir, model_path)
cfg = OmegaConf.load(os.path.join(base_model_path, 'config.yaml'))
cfg.checkpointer.checkpoint_dir = base_model_path
cfg.checkpointer.checkpoint_files = ["torchtune_model_last.pt"]
cfg.train_from_scratch = False
cfg.metric_logger.mode = 'disabled'

recipe = SelfPredictionTrainingRecipeDistributed(cfg=cfg)
recipe.setup(cfg=cfg)

DEBUG:torchtune.utils._logging:Setting manual seed to local seed 1324202648. Local seed is seed + rank = 1324202648 + 0
wandb: ERROR Failed to detect the name of this notebook. You can set it manually with the WANDB_NOTEBOOK_NAME environment variable to enable code saving.


run id:  ma83uh14
None


INFO:torchtune.utils._logging:Model is initialized with precision torch.bfloat16.
INFO:torchtune.utils._logging:Memory stats after model init:
	GPU peak memory allocation: 0.18 GiB
	GPU peak memory reserved: 0.20 GiB
	GPU peak memory active: 0.18 GiB
INFO:torchtune.utils._logging:Optimizer is initialized.
INFO:torchtune.utils._logging:information bottleneck: continuous
INFO:torchtune.utils._logging:phi loss factor: 0.001
INFO:torchtune.utils._logging:self critic loss factor: 0.1
INFO:torchtune.utils._logging:Loss is initialized.
INFO:torchtune.utils._logging:Dataset and Sampler are initialized.
INFO:torchtune.utils._logging:Learning rate scheduler is initialized.
INFO:torchtune.utils._logging: Profiler config after instantiation: {'enabled': False}


In [6]:
@torch.no_grad()
def process_data(
    recipe,
    num_datapoints=10,
    dataset=None,
    batch_size=4,
):
    """
    Processes data from a dataset to generate a specified number of datapoints
    with per-token loss information.

    This function iterates through the dataset, forms batches, calculates
    per-token losses using the `generate_per_token_losses` function,
    adjusts losses related to hidden state predictions by padding,
    and then splits the batch results into individual datapoint dictionaries.

    Args:
        recipe: A recipe object containing model, tokenizer, dataloader (for default dataset), etc.
        num_datapoints (int, optional): The target number of datapoints to generate. Defaults to 10.
        dataset (Dataset, optional): The dataset to process. If None, uses `recipe._dataloader.dataset`.
                                     Defaults to None.
        batch_size (int, optional): The number of samples to process in each batch. Defaults to 4.

    Returns:
        list: A list of dictionaries, where each dictionary represents a datapoint
              containing tokens, decoded tokens, various per-token losses, and other
              relevant information from the original sample, all as numpy arrays.
    """
    if dataset is None:
        dataset = recipe._dataloader.dataset

    packed_dataset = PackedOnTheFlyDataset(dataset,
                                           max_seq_len=dataset.max_sample_length)
    datapoints = []
    finished_processing_dataset = False

    # Iterate through the dataset in batches.
    for idx in range(0, len(dataset), batch_size):
        print(f"Processing batch {len(datapoints)+1}/{num_datapoints}")
        current_batch_samples = []
        for i in range(batch_size):
            # get next item from the packed_dataset
            try:
                sample = next(packed_dataset)
                current_batch_samples.append(sample)
            except StopIteration:
                finished_processing_dataset = True
                break
        model_batch = padded_collate_packed(current_batch_samples)

        # Calculate per-token losses
        per_token_losses = generate_per_token_losses(recipe,
                                                     model_batch)

        for key, value in per_token_losses.items():
            if 'next' in key:
                per_token_losses[key] = torch.cat([torch.zeros_like(value[:, 0:1]), value], dim=1)
            if key == 'next_token_losses':
                per_token_losses[key] = per_token_losses[key][:, :-1]


        # split into datapoints
        current_datapoints = []
        for b in range(len(current_batch_samples)):
            start_idx = 0
            sample = current_batch_samples[b]
            for i, seq_len in enumerate(sample['seq_lens']):
                end_idx = start_idx + seq_len
                current_datapoint = {}
                for key, value in sample.items():
                    if len(value) != len(sample['tokens']):
                        continue
                    current_datapoint[key] = value[start_idx:end_idx].detach().cpu().numpy()
                for key, value in per_token_losses.items():
                    if key == 'tokens':
                        continue
                    if type(value) != torch.Tensor or value.numel() <= 1:
                        continue
                    value = value[b][start_idx:end_idx]
                    if type(value) is torch.Tensor:
                        value = value.detach().cpu().float().numpy()
                    current_datapoint[key] = value

                # valid datapoint only if not all tokens are 0
                is_valid_datapoint = not np.all(current_datapoint['tokens'] == 0)
                if is_valid_datapoint:
                    current_datapoints.append(current_datapoint)
                start_idx = end_idx
        datapoints.extend(current_datapoints)
        if len(datapoints) >= num_datapoints:
            break
        if finished_processing_dataset:
            break
    if len(datapoints) > num_datapoints:
        datapoints = datapoints[:num_datapoints]
    return datapoints


In [7]:
recipe._model.eval()
datapoints = process_data(recipe, 100, ds, 8)

Processing batch 1/100


DEBUG:torchtune.utils._logging:Using flex attention for attention computation since a BlockMask was passed in.


Processing batch 9/100
Processing batch 17/100
Processing batch 25/100
Processing batch 33/100
Processing batch 41/100
Processing batch 49/100
Processing batch 57/100
Processing batch 65/100
Processing batch 73/100
Processing batch 81/100
Processing batch 89/100
Processing batch 97/100


In [8]:
datapoints[0].keys()

dict_keys(['tokens', 'labels', 'input_pos', 'new_language', 'learning_level', 'num_states', 'num_edges', 'vocab_size', 'perturbation', 'next_token_losses', 'latent_losses', 'latent_entropy', 'phi_losses'])

In [9]:
def normalize_data(datapoints, key, num, flip=False):
    # strip away 
    max_seq_len = len(datapoints['tokens'])
    max_idx = min(max_seq_len, num)
    data_t = torch.from_numpy(datapoints[key][1:max_idx-1])
    data_normed = normalize(data_t)
    if flip:
        data_normed = 1-data_normed
    return datapoints['tokens'][1:max_idx-1], data_normed

def normalize(data):
    out = (data-data.min()) / (data.max() - data.min())
    return out

In [10]:
decoded_tokens = decode_token_by_token(datapoints[0]['tokens'], tokenizer)

In [11]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm
import matplotlib.colors
from IPython.display import display, HTML
import seaborn as sns

def colorize_chars(chars, color_array):
    """
    Colorizes a list of single-character strings with forced wrapping for long sequences.

    Args:
        chars (list): A list of single-character strings (tokens).
        color_array (np.ndarray or list): An array of numbers between 0 and 1.

    Returns:
        str: An HTML string with each character colorized, designed to wrap.
    """
    chars = decode_token_by_token(chars, tokenizer)
    cmap = sns.color_palette('coolwarm', as_cmap=True)
    
    # 1. Define the CSS style for the wrapper container.
    #    The key is 'white-space: normal;' to allow the inline-blocks to wrap.
    wrapper_style = "white-space: normal; line-height: 1.5; border: 1px solid #ddd; padding: 5px;"
    
    # 2. Define the template for the individual token.
    template = '<span class="token-barcode" style="color: #333; background-color: {}; display: inline-block; padding: 2px 1px; margin: 0;">{}</span>'
    
    colored_string = ''
    for char, color in zip(chars, color_array):
        color_hex = matplotlib.colors.rgb2hex(cmap(color)[:3])
        
        # Handle space and control characters
        display_char = '&nbsp' if char == ' ' else char
        if len(char) == 1 and char < ' ':
            display_char = repr(char).strip("'")
            
        colored_string += template.format(color_hex, display_char)
        
    # 3. Wrap the entire colored string in a <div> with the wrapping style.
    final_html = f'<div style="{wrapper_style}">{colored_string}</div>'
    
    return final_html

## Sequence learning level

In [12]:
string_len = 2048
dp_idx=0

datapoint = datapoints[dp_idx]
level_chars = colorize_chars(*normalize_data(datapoint, 'learning_level',string_len))
display(HTML(level_chars))

## Sequence entropy

In [13]:
entropy_chars = colorize_chars(*normalize_data(datapoint, 'latent_entropy',string_len, True))
display(HTML(entropy_chars))

## Sequence PHi loss

In [14]:
phi_chars = colorize_chars(*normalize_data(datapoint, 'phi_losses',string_len, True))
display(HTML(phi_chars))

## Sequence NLL 

In [15]:
phi_chars = colorize_chars(*normalize_data(datapoint, 'next_token_losses',string_len, True))
display(HTML(phi_chars))

In [16]:
from modules.architectures import TransformerDecoder
from typing import Optional, List, Tuple, Optional, Callable
# from torchtune.generation._generation import get_causal_mask_from_padding_mask

def get_causal_mask_from_padding_mask(
    padding_mask: torch.Tensor, target_seq_len: Optional[int] = None
) -> torch.Tensor:
    bsz, seq_len = padding_mask.shape
    print('seq len',seq_len)
    target_seq_len = seq_len if target_seq_len is None else target_seq_len
    print('target seq len',target_seq_len)

    if target_seq_len < seq_len:
        # return None
        raise AssertionError(
            "target_seq_len cannot be shorter than the sequence length of the padding mask."
        )

    mask = torch.tril(
        torch.ones(seq_len, target_seq_len, device=padding_mask.device, dtype=bool),
        diagonal=0,
    ).repeat(bsz, 1, 1)
    mask.narrow(2, 0, seq_len).mul_(padding_mask[:, None, :].expand(-1, seq_len, -1))
    mask.diagonal(dim1=1, dim2=2).copy_(torch.Tensor([True]))
    return mask

@torch.inference_mode()
def generate_fn(
    model: TransformerDecoder,
    prompt: torch.Tensor,
    *,
    max_generated_tokens: int,
    pad_id: int = 0,
    temperature: float = 1.0,
    top_k: Optional[int] = None,
    stop_tokens: Optional[List[int]] = None,
    rng: Optional[torch.Generator] = None,
    custom_generate_next_token: Optional[Callable] = None,
) -> Tuple[torch.Tensor, torch.Tensor]:
    prompt = prompt.view(1, -1) if prompt.ndim == 1 else prompt

    if custom_generate_next_token is None:
        custom_generate_next_token = generate_next_token

    bsz, prompt_length = prompt.size()
    total_response_length = prompt_length + max_generated_tokens

    generated_tokens = prompt.clone()
    incremental_decoding = model.caches_are_enabled()

    # grab the correct max_seq_len to generate full causal masks/position ids
    # this is the model's max cache len if incremental decoding, or the sequence
    # length otherwise
    max_seq_len = (
        total_response_length
        if not incremental_decoding
        else model.decoder_max_cache_seq_len
    )

    padding_masks = generated_tokens != pad_id

    if not padding_masks.all():
        # we have padding in the prompt due to varying-length sequences in a batch
        # extend padding masks out to the correct seq len
        padding_masks = torch.nn.functional.pad(
            padding_masks, (0, max_generated_tokens), value=True
        )

        # generate the full causal mask for the whole padding mask with padding ignored
        masks = get_causal_mask_from_padding_mask(
            padding_masks, target_seq_len=max_seq_len
        )

        # right-shift position IDs to account for padding
        input_pos = get_position_ids_from_padding_mask(padding_masks)
    else:
        # just use a regular causal mask if there is no padding
        masks = torch.tril(
            torch.ones(
                total_response_length,
                max_seq_len,
                dtype=torch.bool,
                device=prompt.device,
            )
        ).unsqueeze(0)
        input_pos = torch.arange(
            0, total_response_length, device=generated_tokens.device
        ).unsqueeze(0)

    if incremental_decoding:
        # if KV-caches are enabled, we need a causal mask of shape [bsz, prompt_length, max_cache_len]
        # to match the key/value cache tensor shapes
        curr_masks = masks[:, :prompt_length]
    else:
        # otherwise the causal mask is shape [bsz, prompt_length, prompt_length] because key/value
        # tensors are of identical shape to the prompt
        curr_masks = masks[:, :prompt_length, :prompt_length]

    q = None
    if rng is not None:
        q = torch.empty(
            (bsz, model.tok_embeddings.num_embeddings), device=prompt.device
        ).exponential_(1, generator=rng)
    tokens, generated_logits = generate_next_token(
        model,
        input_pos=input_pos[:, :prompt_length].squeeze(),
        mask=curr_masks,
        x=prompt,
        temperature=temperature,
        top_k=top_k,
        q=q,
    )

    generated_tokens = torch.cat([generated_tokens, tokens], dim=-1)

    curr_pos = prompt_length

    # keeps track at a high level if we've already hit a stop token in a sequence so we can early stop
    stop_token_reached = torch.zeros(bsz, dtype=torch.bool, device=prompt.device)
    stop_tokens = (
        torch.tensor(stop_tokens, device=prompt.device, dtype=tokens.dtype)
        if stop_tokens
        else None
    )

    # everything in stop_token_mask starts as 1s, and we'll set them to 0 for sequences
    # that already hit a stop token
    stop_token_mask = torch.ones(
        (bsz, prompt_length + 1), dtype=torch.int32, device=prompt.device
    )

    # stop early if we reach a stop token in every seq
    if stop_tokens is not None:
        stop_token_reached = update_stop_tokens_tracker(
            tokens, stop_tokens, stop_token_reached
        )
        if stop_token_reached.all().item():
            return generated_tokens, generated_logits

    for _ in range(max_generated_tokens - 1):
        # update stop_token_mask if we reached a stop token in a previous step
        # by appending the logical not of stop_token_reached to the end of the mask
        # reshaped to be bsz first
        if stop_tokens is not None:
            stop_token_mask = torch.cat(
                [stop_token_mask, ~stop_token_reached.reshape(bsz, 1)], dim=-1
            )

        # if incremental decoding is enabled, we can use the current position
        # otherwise, we take the whole sequence up to the current position
        if incremental_decoding:
            curr_input_pos = input_pos[:, curr_pos]
            curr_masks = masks[:, curr_pos, None, :]
        else:
            tokens = generated_tokens.clone()
            curr_input_pos = input_pos[:, : curr_pos + 1]
            curr_masks = masks[:, : curr_pos + 1, : curr_pos + 1]

        q = None
        if rng is not None:
            q = torch.empty(
                (bsz, model.tok_embeddings.num_embeddings), device=prompt.device
            ).exponential_(1, generator=rng)
        tokens, logits = custom_generate_next_token(
            model,
            input_pos=curr_input_pos,
            x=tokens.clone(),
            mask=curr_masks,
            temperature=temperature,
            top_k=top_k,
            q=q,
        )
        generated_tokens = torch.cat([generated_tokens, tokens], dim=-1)
        curr_pos += 1
        if incremental_decoding:
            generated_logits = torch.cat([generated_logits, logits], dim=1)
        else:
            generated_logits = logits

        if stop_tokens is not None:
            stop_token_reached = update_stop_tokens_tracker(
                tokens, stop_tokens, stop_token_reached
            )
            if stop_token_reached.all():
                break

    # mask out generated tokens in seqs that already hit a stop token
    if stop_tokens is not None:
        generated_tokens *= stop_token_mask
        generated_logits *= stop_token_mask[:, :-1, None]

    return generated_tokens, generated_logits

In [17]:
from torchtune import generation

from evaluation.custom_generation import generate_next_token_only_lowercase
from evaluation.pfa_evaluation import recognized_prefix_length

def generate(prompt_tokens,
             recipe,
             max_new_tokens=100,
             return_logits=False,
             temperature=0.6,
             top_k=300,
             stop_tokens=[],
             custom_generate_next_token=None):
    if not type(prompt_tokens[0]) == list:
        prompt_tokens = [prompt_tokens]
    for i, prompt in enumerate(prompt_tokens):
        if prompt[-1] == recipe._tokenizer.eos_id:
            # print("remove eos")
            prompt_tokens[i] = prompt[:-1]
    max_len = max([len(p) for p in prompt_tokens])
    
    # fill with zeros
    print([max_len-len(p) for p in prompt_tokens])
    prompt_tokens = [p + [0] * (max_len - len(p)) for p in prompt_tokens]
    print([len(p) for p in prompt_tokens])
    
    prompt = torch.tensor(prompt_tokens, dtype=torch.int, device=recipe._device)

    generated_tokens, generated_logits = generate_fn(
                model=recipe._model,
                prompt=prompt,
                max_generated_tokens=max_new_tokens,
                pad_id=recipe._tokenizer.pad_id,
                temperature=temperature,
                top_k=top_k,
                stop_tokens=stop_tokens,
                custom_generate_next_token=custom_generate_next_token,
            )
    generated_tokens = generated_tokens[:, 1:]
    print(generated_tokens)

    return_dicts = []
    # generated_tokens = generated_tokens[:, prompt.shape[1]-1:]
    for i in range(generated_tokens.shape[0]):
        gen_tok = generated_tokens[i]
        gen_log = generated_logits[i]
        mask = gen_tok == recipe._tokenizer.pad_id
        gen_tok = gen_tok[~mask]
        gen_log = gen_log[~mask[-gen_log.shape[0]:].to(gen_log.device)]
        decoded_tokens = recipe._tokenizer.decode(gen_tok.tolist())
        #selected_log_probs = gen_log.log_softmax(dim=-1)[range(len(gen_tok)), gen_tok.to(gen_log.device)]
        return_dict = {
            "decoded_tokens": decoded_tokens,
            "generated_tokens": gen_tok,
            "neg_log_probs": gen_log,
        }
        return_dicts.append(return_dict)
    return return_dicts

    # generated_tokens: [bsz x seq_length]
    # generated_logits: [bsz x seq_length x vocab_size]

    selected_neg_log_probs = generated_logits.log_softmax(dim=-1)[
        range(len(generated_tokens[1:])), generated_tokens[1:]
    ]

    decoded_tokens = recipe._tokenizer.decode(generated_tokens.tolist())
    return_dict = {
        "decoded_tokens": decoded_tokens,
        "generated_tokens": generated_tokens,
        "neg_log_probs": selected_neg_log_probs,
    }
    if return_logits:
        return_dict["logits"] = generated_logits
    return return_dict

In [20]:
def evaluate_language_generation_length(recipe,
                                        num_samples=1000,
                                        batch_size=100):
    kwargs = dict(recipe.cfg.dataset)
    kwargs.pop("_component_")
    kwargs["tokenizer"] = tokenizer
    kwargs["word_perturbation_rate"] = 0.
    kwargs["token_perturbation_rate"] = 0.
    kwargs["sequences_per_language_min"] = 10
    kwargs["sequences_per_language_max"] = 11
    kwargs["included_learning_levels"] = [0,1,2,3,4,5]
    dataset = learning_levels_pfa_dataset(**kwargs)

    char_to_idx = {c: i for i, c in enumerate(dataset.letters)}

    with recipe._device:
        recipe._model.setup_caches(
            batch_size=batch_size,
            dtype=recipe._dtype,
            decoder_max_seq_len=2048,
        )
    previous_num_chunks = recipe._model.num_output_chunks
    recipe._model.num_output_chunks = 0

    correct_lengths = []
    for i in range(0, num_samples, batch_size):
        recipe._model.reset_caches()
        recipe._model.self_prediction_losses.reset()
        tokens = []
        transition_symbols = []
        for j in range(batch_size):
            sample = dataset[i]
            tokens.append(sample['tokens'])
            # print(f"{sample['_transition_symbols']}\n")
            print(len(sample['_transition_symbols']))
            transition_symbols.append(np.array(sample['_transition_symbols'][0]))
        generated_sequences = generate(tokens,
                     recipe=recipe,
                     max_new_tokens=100,
                     top_k=18,
                     stop_tokens=[32],
                     custom_generate_next_token=generate_next_token_only_lowercase)
        for j, d in enumerate(generated_sequences):
            decoded_tokens = d['decoded_tokens']
            sequences = decoded_tokens.split(' ')
            sequences_np = [np.array([char_to_idx[c] if c in char_to_idx else 25 for c in s]) for s in sequences]
            current_symbols = transition_symbols[j]

            for seq_i, seq in enumerate(sequences_np):
                len_correct, accepted = recognized_prefix_length(current_symbols, seq)
                if seq_i == len(sequences_np) - 1:
                    # print(seq)
                    # print(f"{i + j + 1}/{num_samples}: {len_correct}/{len(seq)} correct")
                    correct_lengths.append(len_correct)
                else:
                    assert accepted

    correct_lengths = np.array(correct_lengths)
    # ci_mean, ci_lower, ci_higher = bootstrapped_mean_and_ci(correct_lengths, num_samples=10000)

    delete_kv_caches(recipe._model)
    recipe._model.num_output_chunks = previous_num_chunks

    return {
        "accepted_lengths": correct_lengths.tolist(),
        "mean": correct_lengths.mean(),
        "std": correct_lengths.std(),
        "std_err": correct_lengths.std() / np.sqrt(correct_lengths.size),
        # "ci_mean": ci_mean,
        # "ci_95": [ci_lower, ci_higher]
    }

In [21]:
evaluate_language_generation_length(recipe, 1000, 16)

3
2
3
5
2
2
5
4
4
4
4
5
2
4
4
4
[249, 66, 17, 65, 57, 63, 140, 238, 264, 81, 63, 46, 238, 0, 210, 171]
[2046, 2046, 2046, 2046, 2046, 2046, 2046, 2046, 2046, 2046, 2046, 2046, 2046, 2046, 2046, 2046]
seq len 2146
target seq len 2048


AssertionError: target_seq_len cannot be shorter than the sequence length of the padding mask.